In [50]:
from pathlib import Path
import json
import random
import warnings

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import cv2
import pywt
import joblib

import torch
import torch.nn as nn
import torch.nn.functional as F

warnings.filterwarnings("ignore")

# =========================
# Project paths
# =========================
BASE_DIR = Path(r"C:\Users\LYG Y9000x\OneDrive\Desktop\proj")

DATA_ROOT = BASE_DIR / "ecg_dataset" / "development"
WEIGHTS_DIR = BASE_DIR / "weights_from_images_binary"
OUTPUTS_DIR = BASE_DIR / "outputs_binary"
RAG_BASE_DIR = OUTPUTS_DIR / "rag_explainability"

MODEL_NAME = "Original_DSR_Attn_Proto"

# 你的新模型 pth 文件
CKPT_PATH = WEIGHTS_DIR / "Original_DSR_Attn_Proto_best.pth"

# 如果你本地文件还叫旧名，就自动 fallback
if not CKPT_PATH.exists():
    alt = WEIGHTS_DIR / "Original_DSR_Attn_RAG_best.pth"
    if alt.exists():
        CKPT_PATH = alt

# 最终 RAG 输出目录
RAG_DIR = RAG_BASE_DIR / MODEL_NAME
RAG_DIR.mkdir(parents=True, exist_ok=True)

CLASS_NAMES = ["Normal", "Abnormal"]

# 数据集四个原始类别，二分类时 normal=0，其余=1
CLASSES = [
    "normal_ecg_images",
    "abnormal_heartbeat_ecg_images",
    "myocardial_infarction_ecg_images",
    "post_mi_history_ecg_images",
]

IMG_SIZE = 224
USE_CWT = True
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("=" * 90)
print("MODEL_NAME:", MODEL_NAME)
print("BASE_DIR:", BASE_DIR)
print("DATA_ROOT:", DATA_ROOT)
print("WEIGHTS_DIR:", WEIGHTS_DIR)
print("CKPT_PATH:", CKPT_PATH)
print("RAG_DIR:", RAG_DIR)
print("DEVICE:", DEVICE)
print("=" * 90)

assert BASE_DIR.exists(), f"BASE_DIR not found: {BASE_DIR}"
assert DATA_ROOT.exists(), f"DATA_ROOT not found: {DATA_ROOT}"
assert CKPT_PATH.exists(), f"Checkpoint not found: {CKPT_PATH}"

MODEL_NAME: Original_DSR_Attn_Proto
BASE_DIR: C:\Users\LYG Y9000x\OneDrive\Desktop\proj
DATA_ROOT: C:\Users\LYG Y9000x\OneDrive\Desktop\proj\ecg_dataset\development
WEIGHTS_DIR: C:\Users\LYG Y9000x\OneDrive\Desktop\proj\weights_from_images_binary
CKPT_PATH: C:\Users\LYG Y9000x\OneDrive\Desktop\proj\weights_from_images_binary\Original_DSR_Attn_Proto_best.pth
RAG_DIR: C:\Users\LYG Y9000x\OneDrive\Desktop\proj\outputs_binary\rag_explainability\Original_DSR_Attn_Proto
DEVICE: cuda


In [51]:
def load_threshold(model_name: str, default: float = 0.50):
    candidates = [
        WEIGHTS_DIR / "THR_MAP.json",
        OUTPUTS_DIR / "THR_MAP.json",
        OUTPUTS_DIR / "tables" / "THR_MAP.json",
    ]

    possible_keys = [
        model_name,
        "Original_DSR_Attn_RAG",   # 兼容旧名字
    ]

    for p in candidates:
        if not p.exists():
            continue

        try:
            obj = json.loads(p.read_text(encoding="utf-8"))

            for k in possible_keys:
                if k in obj:
                    print(f"Loaded threshold from {p}: {k} = {obj[k]}")
                    return float(obj[k])

        except Exception as e:
            print(f"[Warning] Failed to read threshold JSON: {p}")
            print(type(e).__name__, e)

    print(f"[Info] No threshold found for {model_name}. Use default threshold = {default}")
    return float(default)


THRESHOLD = load_threshold(MODEL_NAME, default=0.50)
print("Final THRESHOLD:", THRESHOLD)

Loaded threshold from C:\Users\LYG Y9000x\OneDrive\Desktop\proj\weights_from_images_binary\THR_MAP.json: Original_DSR_Attn_Proto = 0.05
Final THRESHOLD: 0.05


In [52]:
def generate_cwt_scalogram(signal_1d, scales=None, wavelet="morl", size=(224, 224)):
    if scales is None:
        scales = np.arange(4, 65)

    coeffs, _ = pywt.cwt(signal_1d, scales, wavelet)
    scalogram = np.log1p(np.abs(coeffs))
    scalogram = (scalogram - scalogram.min()) / (scalogram.max() - scalogram.min() + 1e-8)
    scalogram = cv2.resize(scalogram, size, interpolation=cv2.INTER_LINEAR)
    return scalogram.astype(np.float32)


def suppress_grid_gray(gray_u8: np.ndarray) -> np.ndarray:
    g = cv2.GaussianBlur(gray_u8, (3, 3), 0)
    inv = 255 - g

    bw = cv2.adaptiveThreshold(
        inv,
        255,
        cv2.ADAPTIVE_THRESH_MEAN_C,
        cv2.THRESH_BINARY,
        31,
        -5,
    )

    h, w = bw.shape
    kx = max(10, w // 80)
    ky = max(10, h // 80)

    horiz = cv2.morphologyEx(
        bw,
        cv2.MORPH_OPEN,
        cv2.getStructuringElement(cv2.MORPH_RECT, (kx, 1)),
    )

    vert = cv2.morphologyEx(
        bw,
        cv2.MORPH_OPEN,
        cv2.getStructuringElement(cv2.MORPH_RECT, (1, ky)),
    )

    grid = cv2.bitwise_or(horiz, vert)
    grid = cv2.dilate(
        grid,
        cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3)),
        iterations=1,
    )

    out = gray_u8.copy()
    out[grid > 0] = np.clip(out[grid > 0] + 60, 0, 255)
    return out


def extract_ecg_from_image(image_pil, band_ratio: float = 0.55):
    img0 = image_pil.convert("L")
    arr0 = np.array(img0, dtype=np.uint8)
    orig_h, orig_w = arr0.shape[:2]

    # 1) 裁底部 rhythm strip
    rhythm_y0 = int(orig_h * (1 - 0.32))
    rhythm_y1 = orig_h
    rhythm = arr0[rhythm_y0:rhythm_y1, :]

    gray = suppress_grid_gray(rhythm)

    # 2) 左右裁剪
    h_r, w_r = gray.shape
    local_x0 = int(w_r * 0.06)
    local_x1 = int(w_r * 0.94)
    gray = gray[:, local_x0:local_x1]

    h, w = gray.shape

    # 3) 中间 band 裁剪
    y0 = int(h * (0.5 - band_ratio / 2))
    y1 = int(h * (0.5 + band_ratio / 2))
    y0 = max(0, y0)
    y1 = min(h, y1)

    strip = gray[y0:y1, :]
    bh = strip.shape[0]

    inv = 255.0 - strip.astype(np.float32)
    thr = np.percentile(inv, 96)
    wgt = np.clip(inv - thr, 0, None)

    ys = np.arange(bh, dtype=np.float32)[:, None]
    denom = wgt.sum(axis=0)

    eps = 1e-6
    denom_thr = np.percentile(denom, 50)
    valid = denom > denom_thr
    denom_safe = denom + eps

    yhat = (ys * wgt).sum(axis=0) / denom_safe
    yhat[~valid] = np.nan

    x = np.arange(w, dtype=np.float32)
    good = ~np.isnan(yhat)

    if good.sum() >= 2:
        yhat = np.interp(x, x[good], yhat[good])
    else:
        yhat = np.full_like(x, bh / 2.0)

    sig = bh / 2.0 - yhat

    k = 31
    sig = np.convolve(sig, np.ones(k, dtype=np.float32) / k, mode="same")

    k2 = 201
    baseline = np.convolve(sig, np.ones(k2, dtype=np.float32) / k2, mode="same")
    sig = sig - baseline

    lo, hi = np.percentile(sig, [1, 99])
    sig = np.clip(sig, lo, hi)
    sig = (sig - sig.mean()) / (sig.std() + 1e-6)

    # 4) 两端裁掉再重采样
    ww = sig.shape[0]
    l = int(ww * 0.02)
    r = int(ww * 0.98)

    sig_mid = sig[l:r]

    x_old = np.linspace(0, 1, sig_mid.shape[0])
    x_new = np.linspace(0, 1, ww)

    sig = np.interp(x_new, x_old, sig_mid).astype(np.float32)
    return sig


def preprocess_image(image_or_path):
    if isinstance(image_or_path, (str, Path)):
        img_pil = Image.open(image_or_path).convert("L")
    elif isinstance(image_or_path, Image.Image):
        img_pil = image_or_path.convert("L")
    else:
        raise TypeError("image_or_path must be a path or PIL.Image")

    if USE_CWT:
        signal_1d = extract_ecg_from_image(img_pil)
        scalogram = generate_cwt_scalogram(signal_1d, size=(IMG_SIZE, IMG_SIZE))

        # shape: [1, 1, H, W]
        img_tensor = torch.from_numpy(scalogram).unsqueeze(0).unsqueeze(0).float()

        # IMPORTANT: match training-time normalization
        img_tensor = (img_tensor - 0.5) / 0.5

    else:
        arr = np.array(img_pil.resize((IMG_SIZE, IMG_SIZE)), dtype=np.float32) / 255.0
        arr = (arr - 0.5) / 0.5
        img_tensor = torch.from_numpy(arr).unsqueeze(0).unsqueeze(0).float()

    rr_tensor = torch.zeros((1, 0), dtype=torch.float32)
    return img_tensor, rr_tensor

In [53]:
class SE2D(nn.Module):
    def __init__(self, c, r=16):
        super().__init__()
        hidden = max(c // r, 4)
        self.fc1 = nn.Conv2d(c, hidden, 1)
        self.fc2 = nn.Conv2d(hidden, c, 1)

    def forward(self, x):
        s = F.adaptive_avg_pool2d(x, 1)
        s = F.relu(self.fc1(s), inplace=True)
        s = torch.sigmoid(self.fc2(s))
        return x * s


class DSRes2D(nn.Module):
    def __init__(self, in_c, out_c, stride=1):
        super().__init__()

        self.dw = nn.Conv2d(
            in_c,
            in_c,
            3,
            stride=stride,
            padding=1,
            groups=in_c,
            bias=False,
        )

        self.pw = nn.Conv2d(in_c, out_c, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(in_c)
        self.bn2 = nn.BatchNorm2d(out_c)
        self.se = SE2D(out_c)

        self.proj = (
            nn.Conv2d(in_c, out_c, 1, stride=stride, bias=False)
            if (in_c != out_c or stride != 1)
            else None
        )

    def forward(self, x):
        identity = x

        x = F.relu(self.bn1(self.dw(x)), inplace=True)
        x = self.bn2(self.pw(x))
        x = self.se(F.relu(x, inplace=True))

        if self.proj is not None:
            identity = self.proj(identity)

        return F.relu(x + identity, inplace=True)


class MorphologyCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.stem = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(True),
            nn.MaxPool2d(2),
        )

        self.b1 = DSRes2D(32, 64, stride=2)
        self.b2 = DSRes2D(64, 128, stride=2)
        self.b3 = DSRes2D(128, 128, stride=2)

        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(128, 256),
        )

    def forward(self, x):
        x = self.stem(x)
        x = self.b1(x)
        x = self.b2(x)
        x = self.b3(x)
        return self.head(x)


class RR_TCN(nn.Module):
    def __init__(self, rr_dim, out_dim=64):
        super().__init__()
        self.rr_dim = int(rr_dim)

        if self.rr_dim <= 0:
            self.enabled = False
            self.out_dim = 0
            self.net = None
            return

        self.enabled = True
        self.out_dim = out_dim

        self.net = nn.Sequential(
            nn.Linear(self.rr_dim, 128),
            nn.ReLU(True),
            nn.Dropout(0.1),
            nn.Linear(128, out_dim),
            nn.ReLU(True),
        )

    def forward(self, rr):
        if not self.enabled:
            return rr.new_zeros((rr.shape[0], 0))
        return self.net(rr)


class DualBranchSelfAttention(nn.Module):
    def __init__(self, fm_dim=256, fr_dim=64, attn_dim=256, num_heads=4, dropout=0.1):
        super().__init__()

        self.fm_proj = nn.Linear(fm_dim, attn_dim)
        self.fr_proj = nn.Linear(fr_dim, attn_dim) if fr_dim > 0 else None

        self.attn = nn.MultiheadAttention(
            embed_dim=attn_dim,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True,
        )

        self.norm1 = nn.LayerNorm(attn_dim)
        self.norm2 = nn.LayerNorm(attn_dim)

        self.ffn = nn.Sequential(
            nn.Linear(attn_dim, attn_dim),
            nn.ReLU(True),
            nn.Dropout(dropout),
            nn.Linear(attn_dim, attn_dim),
        )

        self.dropout = nn.Dropout(dropout)
        self.has_rr = fr_dim > 0

    def forward(self, fm, fr=None):
        fm_tok = self.fm_proj(fm).unsqueeze(1)

        if self.has_rr and fr is not None and fr.shape[1] > 0:
            fr_tok = self.fr_proj(fr).unsqueeze(1)
            x = torch.cat([fm_tok, fr_tok], dim=1)
        else:
            x = fm_tok

        attn_out, _ = self.attn(x, x, x, need_weights=False)
        x = self.norm1(x + self.dropout(attn_out))

        ffn_out = self.ffn(x)
        x = self.norm2(x + self.dropout(ffn_out))

        x = x.reshape(x.size(0), -1)
        return x


class RAGModule(nn.Module):
    """
    Internal prototype-based feature augmentation.
    In the report/GUI, call this Prototype Fusion, not external RAG.
    """

    def __init__(self, feature_dim=256, num_prototypes=200, num_classes=2):
        super().__init__()

        self.feature_dim = feature_dim
        self.num_prototypes = num_prototypes
        self.num_classes = num_classes

        self.register_buffer("prototypes", torch.randn(num_prototypes, feature_dim))
        self.register_buffer("prototype_labels", torch.zeros(num_prototypes, dtype=torch.long))
        self.register_buffer("prototype_initialized", torch.tensor(False))

        self.fusion = nn.Sequential(
            nn.Linear(feature_dim * 2, feature_dim),
            nn.ReLU(True),
            nn.Dropout(0.1),
            nn.Linear(feature_dim, feature_dim),
        )

        self.attention = nn.Sequential(
            nn.Linear(feature_dim * 2, 128),
            nn.ReLU(True),
            nn.Linear(128, 1),
            nn.Sigmoid(),
        )

    def retrieve(self, query_features, k=5):
        query_norm = F.normalize(query_features, p=2, dim=1)
        proto_norm = F.normalize(self.prototypes, p=2, dim=1)

        similarities = torch.mm(query_norm, proto_norm.t())
        top_k_sim, top_k_idx = similarities.topk(k, dim=1)

        retrieved_features = self.prototypes[top_k_idx]
        retrieved_labels = self.prototype_labels[top_k_idx]

        return retrieved_features, retrieved_labels, top_k_sim

    def forward(self, query_features):
        if not bool(self.prototype_initialized):
            return query_features

        retrieved_features, retrieved_labels, similarities = self.retrieve(query_features, k=5)

        weights = F.softmax(similarities * 10, dim=1).unsqueeze(2)
        aggregated = (retrieved_features * weights).sum(dim=1)

        combined = torch.cat([query_features, aggregated], dim=1)

        alpha = self.attention(combined)
        fused = self.fusion(combined)

        augmented_features = query_features + alpha * fused
        return augmented_features


class OriginalDSRAblation(nn.Module):
    """
    Original_DSR_Attn_Proto:
    Original DSR + Self-Attention + Prototype Fusion.
    """

    def __init__(self, num_classes=2, rr_dim=0, use_attn=True, use_rag=True):
        super().__init__()

        self.use_attn = bool(use_attn)
        self.use_rag = bool(use_rag)

        self.morph = MorphologyCNN()
        self.rrnet = RR_TCN(rr_dim)

        if self.use_attn:
            self.self_attn_fusion = DualBranchSelfAttention(
                fm_dim=256,
                fr_dim=self.rrnet.out_dim,
                attn_dim=256,
                num_heads=4,
                dropout=0.1,
            )

            attn_out_dim = 256 * 2 if self.rrnet.out_dim > 0 else 256

            self.fuse = nn.Sequential(
                nn.Linear(attn_out_dim, 256),
                nn.ReLU(True),
                nn.Dropout(0.2),
            )

        else:
            fuse_in = 256 + self.rrnet.out_dim

            self.fuse = nn.Sequential(
                nn.Linear(fuse_in, 256),
                nn.ReLU(True),
                nn.Dropout(0.2),
            )

        if self.use_rag:
            self.rag = RAGModule(
                feature_dim=256,
                num_prototypes=200,
                num_classes=num_classes,
            )

        self.cls = nn.Linear(256, num_classes)

    def extract_features(self, img, rr):
        fm = self.morph(img)
        fr = self.rrnet(rr)

        if self.use_attn:
            z = self.self_attn_fusion(fm, fr)
        else:
            z = torch.cat([fm, fr], dim=1)

        z = self.fuse(z)
        return z

    def forward(self, img, rr):
        z = self.extract_features(img, rr)

        if self.use_rag:
            z = self.rag(z)

        return self.cls(z)


print("Model classes are ready.")

Model classes are ready.


In [54]:
def load_checkpoint(ckpt_path, map_location="cpu"):
    try:
        obj = torch.load(
            ckpt_path,
            map_location=map_location,
            weights_only=True,
        )
    except Exception as e:
        print("[Info] weights_only=True failed. Loading trusted local checkpoint with weights_only=False.")
        print("Reason:", type(e).__name__, str(e)[:300])

        obj = torch.load(
            ckpt_path,
            map_location=map_location,
            weights_only=False,
        )

    return obj


ckpt = load_checkpoint(CKPT_PATH, map_location=DEVICE)

print("Checkpoint type:", type(ckpt))
print("Checkpoint keys:", ckpt.keys())

state_dict = ckpt["model_state_dict"]
config = ckpt.get("config", {})

print("Checkpoint config:", config)
print("best_val_f1:", ckpt.get("best_val_f1", None))

use_attn = bool(config.get("use_attn", True))
use_rag = bool(config.get("use_rag", True))
rr_dim = int(config.get("rr_dim", 0))
num_classes = int(config.get("num_classes", 2))

model = OriginalDSRAblation(
    num_classes=num_classes,
    rr_dim=rr_dim,
    use_attn=use_attn,
    use_rag=use_rag,
)

missing, unexpected = model.load_state_dict(state_dict, strict=False)

print("use_attn:", use_attn)
print("use_rag:", use_rag)
print("rr_dim:", rr_dim)
print("num_classes:", num_classes)
print("missing:", len(missing))
print("unexpected:", len(unexpected))

if missing:
    print("missing sample:", missing[:20])

if unexpected:
    print("unexpected sample:", unexpected[:20])

model.to(DEVICE)
model.eval()

assert len(missing) == 0, "There are missing keys. Model definition may not match checkpoint."
assert len(unexpected) == 0, "There are unexpected keys. Model definition may not match checkpoint."

print("Model loaded successfully.")

[Info] weights_only=True failed. Loading trusted local checkpoint with weights_only=False.
Reason: UnpicklingError Weights only load failed. This file can still be loaded, to do so you have two options, do those steps only if you trust the source of the checkpoint. 
	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.l
Checkpoint type: <class 'dict'>
Checkpoint keys: dict_keys(['model_state_dict', 'best_val_f1', 'config', 'history'])
Checkpoint config: {'use_attn': True, 'use_rag': True, 'rr_dim': 0, 'num_classes': 2}
best_val_f1: 0.976
use_attn: True
use_rag: True
rr_dim: 0
num_classes: 2
missing: 0
unexpected: 0
Model loaded successfully.


In [55]:
EXTS = ("*.png", "*.jpg", "*.jpeg", "*.bmp", "*.tif", "*.tiff")


def map_to_binary_label(folder_name: str) -> int:
    if folder_name == "normal_ecg_images":
        return 0
    return 1


def list_image_items(data_root: Path):
    items = []

    for cname in CLASSES:
        cdir = data_root / cname

        if not cdir.exists():
            print("[Warning] Missing class folder:", cdir)
            continue

        for ext in EXTS:
            for p in cdir.rglob(ext):
                y = map_to_binary_label(cname)
                items.append((p, y, cname))

    return items


items = list_image_items(DATA_ROOT)

print("Number of images:", len(items))

label_counts = pd.Series([y for _, y, _ in items]).value_counts().sort_index()
print("Binary label counts:")
print(label_counts)

folder_counts = pd.Series([c for _, _, c in items]).value_counts()
print("\nFolder counts:")
print(folder_counts)

assert len(items) > 0, "No images found. Check DATA_ROOT and class folders."

Number of images: 918
Binary label counts:
0    280
1    638
Name: count, dtype: int64

Folder counts:
normal_ecg_images                   280
myocardial_infarction_ecg_images    238
abnormal_heartbeat_ecg_images       230
post_mi_history_ecg_images          170
Name: count, dtype: int64


In [56]:
@torch.no_grad()
def predict_probs_and_logits(model, img, rr):
    model.eval()
    logits = model(img, rr)
    probs = F.softmax(logits.float(), dim=1)

    return (
        probs[:, 0],   # prob_normal
        probs[:, 1],   # prob_abnormal
        logits[:, 0],  # logit_normal
        logits[:, 1],  # logit_abnormal
    )


@torch.no_grad()
def extract_embedding(model, img, rr, include_internal_proto=True):
    """
    Return 256-D normalized embedding for external RAG retrieval.

    For Original_DSR_Attn_Proto:
    - extract_features() includes self-attention fusion
    - include_internal_proto=True applies internal prototype fusion
    """
    model.eval()

    z = model.extract_features(img, rr)

    if include_internal_proto and getattr(model, "use_rag", False) and hasattr(model, "rag"):
        z = model.rag(z)

    z = F.normalize(z.float(), p=2, dim=1)
    return z


rows = []
embeddings = []

for path, true_label, folder_name in tqdm(items, desc=f"Extracting RAG embeddings for {MODEL_NAME}"):
    try:
        img, rr = preprocess_image(path)

        img = img.to(DEVICE)
        rr = rr.to(DEVICE)

        prob_normal_t, prob_abnormal_t, logit_normal_t, logit_abnormal_t = predict_probs_and_logits(model, img, rr)

        prob_normal = float(prob_normal_t[0].item())
        prob_abnormal = float(prob_abnormal_t[0].item())
        logit_normal = float(logit_normal_t[0].item())
        logit_abnormal = float(logit_abnormal_t[0].item())

        # 与训练 notebook 一致的预测方式
        pred_argmax = int(np.argmax([prob_normal, prob_abnormal]))

        # threshold-based prediction, for report/RAG operating point only
        pred_thr = int(prob_abnormal >= THRESHOLD)

        # 建议先用 argmax 检查模型是否恢复正常
        pred_label = pred_argmax

        emb = extract_embedding(
            model,
            img,
            rr,
            include_internal_proto=True,
        )[0].detach().cpu().numpy().astype("float32")

        rel_path = str(path.relative_to(DATA_ROOT))

        rows.append({
            "source_path": rel_path,
            "folder_name": folder_name,
            "true_label": int(true_label),
            "true_label_name": CLASS_NAMES[int(true_label)],

            "pred_label": int(pred_label),
            "pred_label_name": CLASS_NAMES[int(pred_label)],

            "pred_argmax": int(pred_argmax),
            "pred_thr": int(pred_thr),

            "prob_normal": float(prob_normal),
            "prob_abnormal": float(prob_abnormal),

            "logit_normal": float(logit_normal),
            "logit_abnormal": float(logit_abnormal),

            "threshold": float(THRESHOLD),
            "margin_to_thr": float(abs(prob_abnormal - THRESHOLD)),
            "model_name": MODEL_NAME,
        })

        embeddings.append(emb)

    except Exception as e:
        print("[Warning] Failed:", path)
        print(type(e).__name__, e)

cases_df = pd.DataFrame(rows)
embeddings = np.vstack(embeddings).astype("float32")

print("cases_df shape:", cases_df.shape)
print("embeddings shape:", embeddings.shape)

display(cases_df.head())

print("\nPrediction table:")
print(cases_df[["true_label", "pred_label"]].value_counts().sort_index())

print("\nProbability summary by true label:")
display(cases_df.groupby("true_label")["prob_abnormal"].describe())

meta_path = RAG_DIR / "rag_meta.csv"
emb_path = RAG_DIR / "rag_embeddings.npy"

cases_df.to_csv(meta_path, index=False, encoding="utf-8-sig")
np.save(emb_path, embeddings)

print("Saved:", meta_path)
print("Saved:", emb_path)

Extracting RAG embeddings for Original_DSR_Attn_Proto: 100%|██████████| 918/918 [00:33<00:00, 27.50it/s]

cases_df shape: (918, 15)
embeddings shape: (918, 256)


,source_path,folder_name,true_label,true_label_name,pred_label,pred_label_name,pred_argmax,pred_thr,prob_normal,prob_abnormal,logit_normal,logit_abnormal,threshold,margin_to_thr,model_name
0,normal_ecg_images\Normal(1).jpg,normal_ecg_images,0,Normal,0,Normal,0,1,0.946123,0.053877,1.623189,-1.242484,0.05,0.003877,Original_DSR_Attn_Proto
1,normal_ecg_images\Normal(10).jpg,normal_ecg_images,0,Normal,0,Normal,0,0,0.952334,0.047666,1.704886,-1.289809,0.05,0.002334,Original_DSR_Attn_Proto
2,normal_ecg_images\Normal(100).jpg,normal_ecg_images,0,Normal,0,Normal,0,0,0.960457,0.039543,1.823397,-1.366612,0.05,0.010457,Original_DSR_Attn_Proto
3,normal_ecg_images\Normal(101).jpg,normal_ecg_images,0,Normal,0,Normal,0,1,0.943323,0.056677,1.583895,-1.228152,0.05,0.006677,Original_DSR_Attn_Proto
4,normal_ecg_images\Normal(102).jpg,normal_ecg_images,0,Normal,0,Normal,0,1,0.876710,0.123290,1.090917,-0.870720,0.05,0.073290,Original_DSR_Attn_Proto



Prediction table:
true_label  pred_label
0           0             275
            1               5
1           0               9
            1             629
Name: count, dtype: int64

Probability summary by true label:


,count,mean,std,min,25%,50%,75%,max
true_label,,,,,,,,
0,280.0,0.089342,0.123495,0.033382,0.047523,0.056724,0.081023,0.959636
1,638.0,0.938198,0.091885,0.059010,0.950917,0.957460,0.961321,0.966638


Saved: C:\Users\LYG Y9000x\OneDrive\Desktop\proj\outputs_binary\rag_explainability\Original_DSR_Attn_Proto\rag_meta.csv
Saved: C:\Users\LYG Y9000x\OneDrive\Desktop\proj\outputs_binary\rag_explainability\Original_DSR_Attn_Proto\rag_embeddings.npy


In [57]:
def build_retrieval_index(embeddings: np.ndarray, save_dir: Path):
    embeddings = embeddings.astype("float32")

    try:
        import faiss

        emb_norm = embeddings.copy()
        faiss.normalize_L2(emb_norm)

        index = faiss.IndexFlatIP(emb_norm.shape[1])
        index.add(emb_norm)

        index_path = save_dir / "rag_index.faiss"
        faiss.write_index(index, str(index_path))

        print("FAISS index saved:", index_path)
        return "faiss", index_path

    except Exception as e:
        print("[Info] FAISS unavailable or failed. Fallback to sklearn.")
        print("Reason:", type(e).__name__, str(e)[:300])

        from sklearn.neighbors import NearestNeighbors

        nn = NearestNeighbors(metric="cosine")
        nn.fit(embeddings)

        index_path = save_dir / "rag_index_sklearn.joblib"
        joblib.dump(nn, index_path)

        print("sklearn index saved:", index_path)
        return "sklearn", index_path


index_type, index_path = build_retrieval_index(embeddings, RAG_DIR)

manifest = {
    "model_name": MODEL_NAME,
    "checkpoint_path": str(CKPT_PATH),
    "data_root": str(DATA_ROOT),
    "rag_dir": str(RAG_DIR),
    "num_cases": int(len(cases_df)),
    "embedding_shape": list(embeddings.shape),
    "index_type": index_type,
    "index_path": str(index_path),
    "meta_path": str(meta_path),
    "embedding_path": str(emb_path),
    "threshold": float(THRESHOLD),
    "use_attn": bool(use_attn),
    "use_internal_proto": bool(use_rag),
}

manifest_path = RAG_DIR / "manifest.json"
manifest_path.write_text(
    json.dumps(manifest, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print("Saved manifest:", manifest_path)
display(pd.DataFrame([manifest]))

FAISS index saved: C:\Users\LYG Y9000x\OneDrive\Desktop\proj\outputs_binary\rag_explainability\Original_DSR_Attn_Proto\rag_index.faiss
Saved manifest: C:\Users\LYG Y9000x\OneDrive\Desktop\proj\outputs_binary\rag_explainability\Original_DSR_Attn_Proto\manifest.json


,model_name,checkpoint_path,data_root,rag_dir,num_cases,embedding_shape,index_type,index_path,meta_path,embedding_path,threshold,use_attn,use_internal_proto
0,Original_DSR_Attn_Proto,C:\Users\LYG Y9000x\OneDrive\Desktop\proj\weig...,C:\Users\LYG Y9000x\OneDrive\Desktop\proj\ecg_...,C:\Users\LYG Y9000x\OneDrive\Desktop\proj\outp...,918,"[918, 256]",faiss,C:\Users\LYG Y9000x\OneDrive\Desktop\proj\outp...,C:\Users\LYG Y9000x\OneDrive\Desktop\proj\outp...,C:\Users\LYG Y9000x\OneDrive\Desktop\proj\outp...,0.05,True,True


In [58]:
def load_retrieval_index(index_type: str, index_path: Path):
    if index_type == "faiss":
        import faiss
        return faiss.read_index(str(index_path))

    return joblib.load(index_path)


def retrieve_topk_unique(query_embedding, meta_df, index_obj, index_type="faiss", k=5, overfetch=20):
    query_embedding = np.asarray(query_embedding, dtype="float32").reshape(1, -1)
    overfetch = max(overfetch, k)

    if index_type == "faiss":
        import faiss

        q = query_embedding.copy()
        faiss.normalize_L2(q)

        scores, ids = index_obj.search(q, min(overfetch, len(meta_df)))
        ids = ids[0]
        scores = scores[0]

    else:
        n_neighbors = min(overfetch, len(meta_df))
        dist, ids = index_obj.kneighbors(query_embedding, n_neighbors=n_neighbors)

        ids = ids[0]
        scores = 1.0 - dist[0]

    seen_paths = set()
    keep_rows = []

    for idx, score in zip(ids, scores):
        idx = int(idx)

        if idx < 0 or idx >= len(meta_df):
            continue

        row = meta_df.iloc[idx].to_dict()
        src = str(row["source_path"])

        if src in seen_paths:
            continue

        seen_paths.add(src)
        row["similarity"] = float(score)
        keep_rows.append(row)

        if len(keep_rows) >= k:
            break

    out = pd.DataFrame(keep_rows).reset_index(drop=True)

    if len(out) > 0:
        out.insert(0, "rank", np.arange(1, len(out) + 1))

    return out


meta_df = pd.read_csv(meta_path)
index_obj = load_retrieval_index(index_type, index_path)

test_retrieved = retrieve_topk_unique(
    query_embedding=embeddings[0],
    meta_df=meta_df,
    index_obj=index_obj,
    index_type=index_type,
    k=5,
    overfetch=20,
)

display(test_retrieved)

,rank,source_path,folder_name,true_label,true_label_name,pred_label,pred_label_name,pred_argmax,pred_thr,prob_normal,prob_abnormal,logit_normal,logit_abnormal,threshold,margin_to_thr,model_name,similarity
0,1,normal_ecg_images\Normal(143).jpg,normal_ecg_images,0,Normal,0,Normal,0,1,0.946123,0.053877,1.623189,-1.242484,0.05,0.003877,Original_DSR_Attn_Proto,1.000000
1,2,normal_ecg_images\Normal(1).jpg,normal_ecg_images,0,Normal,0,Normal,0,1,0.946123,0.053877,1.623189,-1.242484,0.05,0.003877,Original_DSR_Attn_Proto,1.000000
2,3,normal_ecg_images\Normal(57).jpg,normal_ecg_images,0,Normal,0,Normal,0,1,0.946366,0.053634,1.633049,-1.237389,0.05,0.003634,Original_DSR_Attn_Proto,0.999924
3,4,normal_ecg_images\Normal(199).jpg,normal_ecg_images,0,Normal,0,Normal,0,1,0.946366,0.053634,1.633049,-1.237389,0.05,0.003634,Original_DSR_Attn_Proto,0.999924
4,5,normal_ecg_images\Normal(153).jpg,normal_ecg_images,0,Normal,0,Normal,0,1,0.946287,0.053713,1.631320,-1.237568,0.05,0.003713,Original_DSR_Attn_Proto,0.999924


In [59]:
def build_text_explanation(model_name, row, threshold, retrieved_df, case_title="Case"):
    pred_label = int(row["pred_label"])
    true_label = int(row["true_label"])
    prob_abnormal = float(row["prob_abnormal"])

    pred_name = CLASS_NAMES[pred_label] if pred_label in [0, 1] else str(pred_label)
    true_name = CLASS_NAMES[true_label] if true_label in [0, 1] else str(true_label)

    if len(retrieved_df) > 0:
        same_pred_ratio = float((retrieved_df["true_label"].values == pred_label).mean())
        same_true_ratio = float((retrieved_df["true_label"].values == true_label).mean())
        abnormal_ratio = float((retrieved_df["true_label"].values == 1).mean())
    else:
        same_pred_ratio = 0.0
        same_true_ratio = 0.0
        abnormal_ratio = 0.0

    if abs(prob_abnormal - threshold) < 0.05:
        conf_note = "This sample is close to the operating threshold, so the decision is borderline."
    elif prob_abnormal >= 0.80 or prob_abnormal <= 0.20:
        conf_note = "The probability is far from the threshold, so the decision confidence is relatively strong."
    else:
        conf_note = "The decision confidence is moderate."

    if same_pred_ratio >= 0.8:
        retrieval_note = "Most retrieved reference cases are consistent with the predicted label."
    elif same_pred_ratio >= 0.5:
        retrieval_note = "The retrieved evidence is partially consistent with the predicted label."
    else:
        retrieval_note = "The retrieved evidence is mixed and does not strongly support the predicted label."

    correctness_note = (
        "The prediction is correct."
        if pred_label == true_label
        else "The prediction is incorrect, so this case is useful for error analysis."
    )

    txt = (
        f"{case_title}\n"
        f"Model: {model_name}\n"
        f"Source: {row['source_path']}\n"
        f"True label: {true_name}\n"
        f"Predicted label: {pred_name}\n"
        f"Probability of Abnormal: {prob_abnormal:.6f}\n"
        f"Decision Threshold: {threshold:.4f}\n"
        f"Retrieved abnormal-case ratio: {abnormal_ratio:.2%}\n"
        f"Retrieved support ratio for predicted label: {same_pred_ratio:.2%}\n"
        f"Retrieved support ratio for true label: {same_true_ratio:.2%}\n"
        f"{correctness_note}\n"
        f"{conf_note}\n"
        f"{retrieval_note}\n"
    )

    return txt

In [60]:
def pick_one_case(df, condition, sort_col, ascending=True, fallback_condition=None, name="case"):
    sub = df[condition].copy()

    exact_found = True

    if len(sub) == 0 and fallback_condition is not None:
        print(f"[Warning] No exact {name} found. Using fallback condition.")
        sub = df[fallback_condition].copy()
        exact_found = False

    if len(sub) == 0:
        print(f"[Warning] No available sample for {name}.")
        return None, False

    row = sub.sort_values(sort_col, ascending=ascending).iloc[0]
    return row, exact_found


case_abn, case_abn_exact = pick_one_case(
    df=cases_df,
    condition=(cases_df.true_label == 1) & (cases_df.pred_label == 1),
    sort_col="prob_abnormal",
    ascending=False,
    fallback_condition=(cases_df.true_label == 1),
    name="correct abnormal case",
)

case_nor, case_nor_exact = pick_one_case(
    df=cases_df,
    condition=(cases_df.true_label == 0) & (cases_df.pred_label == 0),
    sort_col="prob_abnormal",
    ascending=True,
    fallback_condition=(cases_df.true_label == 0),
    name="correct normal case",
)

case_border, case_border_exact = pick_one_case(
    df=cases_df,
    condition=cases_df["margin_to_thr"].notna(),
    sort_col="margin_to_thr",
    ascending=True,
    fallback_condition=None,
    name="borderline case",
)

print("correct abnormal exact:", case_abn_exact)
display(case_abn)

print("correct normal exact:", case_nor_exact)
display(case_nor)

print("borderline exact:", case_border_exact)
display(case_border)

correct abnormal exact: True


source_path        abnormal_heartbeat_ecg_images\HB(192).jpg
folder_name                    abnormal_heartbeat_ecg_images
true_label                                                 1
true_label_name                                     Abnormal
pred_label                                                 1
pred_label_name                                     Abnormal
pred_argmax                                                1
pred_thr                                                   1
prob_normal                                         0.033362
prob_abnormal                                       0.966638
logit_normal                                       -1.653034
logit_abnormal                                       1.71337
threshold                                               0.05
margin_to_thr                                       0.916638
model_name                           Original_DSR_Attn_Proto
Name: 381, dtype: object

correct normal exact: True


source_path        normal_ecg_images\Normal(64).jpg
folder_name                       normal_ecg_images
true_label                                        0
true_label_name                              Normal
pred_label                                        0
pred_label_name                              Normal
pred_argmax                                       0
pred_thr                                          0
prob_normal                                0.966618
prob_abnormal                              0.033382
logit_normal                               1.926552
logit_abnormal                            -1.439249
threshold                                      0.05
margin_to_thr                              0.016618
model_name                  Original_DSR_Attn_Proto
Name: 241, dtype: object

borderline exact: True


source_path        normal_ecg_images\Normal(119).jpg
folder_name                        normal_ecg_images
true_label                                         0
true_label_name                               Normal
pred_label                                         0
pred_label_name                               Normal
pred_argmax                                        0
pred_thr                                           1
prob_normal                                 0.949966
prob_abnormal                               0.050034
logit_normal                                1.681121
logit_abnormal                             -1.262593
threshold                                       0.05
margin_to_thr                               0.000034
model_name                   Original_DSR_Attn_Proto
Name: 22, dtype: object

In [61]:
def save_demo_case(case_row, case_key, case_title, exact_found=True, k=5):
    if case_row is None:
        print(f"[Skip] {case_key}: no case row.")
        return None, None

    source_path = str(case_row["source_path"])

    matched_idx = cases_df.index[cases_df["source_path"] == source_path].tolist()

    if len(matched_idx) == 0:
        print(f"[Skip] {case_key}: source_path not found in cases_df.")
        return None, None

    idx = matched_idx[0]
    query_embedding = embeddings[idx]

    retrieved_df = retrieve_topk_unique(
        query_embedding=query_embedding,
        meta_df=meta_df,
        index_obj=index_obj,
        index_type=index_type,
        k=k,
        overfetch=max(20, k),
    )

    # 如果是 fallback case，在 explanation 里说明一下
    title = case_title
    if not exact_found:
        title = (
            f"{case_title}\n"
            f"Selection note: no exact case was found under the requested condition, "
            f"so this file uses the closest fallback sample."
        )

    explanation = build_text_explanation(
        model_name=MODEL_NAME,
        row=case_row,
        threshold=THRESHOLD,
        retrieved_df=retrieved_df,
        case_title=title,
    )

    retrieved_path = RAG_DIR / f"{case_key}_retrieved.csv"
    explanation_path = RAG_DIR / f"{case_key}_explanation.txt"

    retrieved_df.to_csv(retrieved_path, index=False, encoding="utf-8-sig")
    explanation_path.write_text(explanation, encoding="utf-8")

    print("Saved:", retrieved_path)
    print("Saved:", explanation_path)

    return retrieved_df, explanation


correct_abnormal_retrieved, correct_abnormal_explanation = save_demo_case(
    case_row=case_abn,
    case_key="correct_abnormal",
    case_title="Correct Abnormal Case",
    exact_found=case_abn_exact,
    k=5,
)

correct_normal_retrieved, correct_normal_explanation = save_demo_case(
    case_row=case_nor,
    case_key="correct_normal",
    case_title="Correct Normal Case",
    exact_found=case_nor_exact,
    k=5,
)

borderline_retrieved, borderline_explanation = save_demo_case(
    case_row=case_border,
    case_key="borderline_case",
    case_title="Borderline Case",
    exact_found=case_border_exact,
    k=5,
)

print("\nFinal files under RAG_DIR:")
for p in sorted(RAG_DIR.iterdir()):
    print(" -", p.name)

Saved: C:\Users\LYG Y9000x\OneDrive\Desktop\proj\outputs_binary\rag_explainability\Original_DSR_Attn_Proto\correct_abnormal_retrieved.csv
Saved: C:\Users\LYG Y9000x\OneDrive\Desktop\proj\outputs_binary\rag_explainability\Original_DSR_Attn_Proto\correct_abnormal_explanation.txt
Saved: C:\Users\LYG Y9000x\OneDrive\Desktop\proj\outputs_binary\rag_explainability\Original_DSR_Attn_Proto\correct_normal_retrieved.csv
Saved: C:\Users\LYG Y9000x\OneDrive\Desktop\proj\outputs_binary\rag_explainability\Original_DSR_Attn_Proto\correct_normal_explanation.txt
Saved: C:\Users\LYG Y9000x\OneDrive\Desktop\proj\outputs_binary\rag_explainability\Original_DSR_Attn_Proto\borderline_case_retrieved.csv
Saved: C:\Users\LYG Y9000x\OneDrive\Desktop\proj\outputs_binary\rag_explainability\Original_DSR_Attn_Proto\borderline_case_explanation.txt

Final files under RAG_DIR:
 - borderline_case_explanation.txt
 - borderline_case_retrieved.csv
 - correct_abnormal_explanation.txt
 - correct_abnormal_retrieved.csv
 - cor

In [62]:
required_files = [
    "rag_meta.csv",
    "rag_embeddings.npy",
    "correct_abnormal_explanation.txt",
    "correct_abnormal_retrieved.csv",
    "correct_normal_explanation.txt",
    "correct_normal_retrieved.csv",
    "borderline_case_explanation.txt",
    "borderline_case_retrieved.csv",
]

if index_type == "faiss":
    required_files.append("rag_index.faiss")
else:
    required_files.append("rag_index_sklearn.joblib")

missing_files = []

for fname in required_files:
    p = RAG_DIR / fname
    if not p.exists():
        missing_files.append(fname)

print("RAG_DIR:", RAG_DIR)

if missing_files:
    print("[Warning] Missing files:")
    for f in missing_files:
        print(" -", f)
else:
    print("All required RAG files were generated successfully.")

print("\nFiles:")
for p in sorted(RAG_DIR.iterdir()):
    print(p.name)

RAG_DIR: C:\Users\LYG Y9000x\OneDrive\Desktop\proj\outputs_binary\rag_explainability\Original_DSR_Attn_Proto
All required RAG files were generated successfully.

Files:
borderline_case_explanation.txt
borderline_case_retrieved.csv
correct_abnormal_explanation.txt
correct_abnormal_retrieved.csv
correct_normal_explanation.txt
correct_normal_retrieved.csv
manifest.json
rag_embeddings.npy
rag_index.faiss
rag_meta.csv
